# pyfixest + MLflow tracking example

This notebook makes some synthetic data, runs it through `regress` (which wraps a pyfixest modeling function and logs to MLflow), and then pulls the logged runs back out to inspect them.

In [ ]:
import warnings

# keep outputs reproducible: this warning embeds an absolute path
warnings.filterwarnings("ignore", message="IProgress not found")

import logging
import os

# artifact-download progress bars print throughput rates -> not reproducible
os.environ["MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR"] = "false"

import mlflow
import numpy as np
import pandas as pd
from IPython.display import Markdown

from pyfixest_regression import coeftable, etable, regress, results_table

# mlflow's INFO logs carry timestamps; silence them so outputs are stable
logging.getLogger("mlflow").setLevel(logging.ERROR)

mlflow.set_tracking_uri("sqlite:///mlflow.db")
# Set the experiment once. regress then logs into the active experiment
# (you could also pass experiment_name= per call).
_ = mlflow.set_experiment("pyfixest-example")

## Make some data

A small treatment-effect DGP: a binary `treat`, two covariates `age` and `income`, and `outcome = 1 + 2*treat - 0.03*age + 0.5*income + noise`. The estimand of interest is the coefficient on `treat`.

In [ ]:
rng = np.random.default_rng(0)
n = 500

treat = rng.integers(0, 2, size=n)  # binary treatment (0/1)
age = rng.normal(50, 10, size=n)
income = rng.normal(size=n)
outcome = 1.0 + 2.0 * treat - 0.03 * age + 0.5 * income + rng.normal(size=n)

data = pd.DataFrame({"outcome": outcome, "treat": treat, "age": age, "income": income})
data.head()

,outcome,treat,age,income
0,0.741589,1,51.992180,-0.670439
1,1.163218,1,46.179977,-0.690944
2,2.752725,1,75.524240,-1.446884
3,1.009722,0,46.755281,0.754386
4,-1.107206,0,37.787767,-0.395864


## Run it

Two runs on the same data — one with `iid` errors, one with `hetero` (heteroskedasticity-robust) errors — each given a `name` so we can pick it out again later. `vcov` is part of each run's content hash, so these are two distinct runs rather than one deduplicated run.

In [ ]:
fit_iid = regress("outcome ~ treat + age + income", data=data, vcov="iid", name="iid")
fit_hetero = regress(
    "outcome ~ treat + age + income", data=data, vcov="hetero", name="hetero"
)

fit_iid.summary()

###

Estimation:  OLS
Dep. var.: outcome
sample: None = all
Inference:  iid
Observations:  500

| Coefficient   |   Estimate |   Std. Error |   t value |   Pr(>|t|) |   2.5% |   97.5% |
|:--------------|-----------:|-------------:|----------:|-----------:|-------:|--------:|
| Intercept     |      0.483 |        0.247 |     1.954 |      0.051 | -0.003 |   0.968 |
| treat         |      1.978 |        0.093 |    21.218 |      0.000 |  1.795 |   2.162 |
| age           |     -0.019 |        0.005 |    -3.999 |      0.000 | -0.028 |  -0.010 |
| income        |      0.493 |        0.048 |    10.173 |      0.000 |  0.398 |   0.588 |
---
RMSE: 1.029 R2: 0.542 


## Get the runs back

`regress` returns the fitted model directly, but everything it logged is also in MLflow. `results_table` wraps `mlflow.search_runs` and returns a tidy one-row-per-run frame (params and metrics, prefixes stripped). Alongside the usual fit metrics it records a short summary of each fit — `n_coefs`, `n_fes` (number of absorbed fixed effects), and `estimation_time` (seconds).

In [ ]:
runs = results_table("pyfixest-example")
runs[["fml", "vcov", "nobs", "n_coefs", "n_fes", "r2", "adj_r2"]]

,fml,vcov,nobs,n_coefs,n_fes,r2,adj_r2
0,outcome ~ treat + age + income,hetero,500.0,4.0,0.0,0.541811,0.539039
1,outcome ~ treat + age + income,iid,500.0,4.0,0.0,0.541811,0.539039


### Filter the runs

`results_table` forwards a `filter_string` straight to MLflow, so you can pull just the runs you want — by a run's `name` (stored as the `mlflow.runName` tag), a metric, or a param.

In [ ]:
results_table("pyfixest-example", filter_string="tags.`mlflow.runName` = 'hetero'")[
    ["fml", "vcov", "nobs", "r2"]
]

,fml,vcov,nobs,r2
0,outcome ~ treat + age + income,hetero,500.0,0.541811


## Key coefficients as searchable metrics

Every run also logs its first few coefficients as plain numeric metrics — `coef.<name>`, `se.<name>`, `pvalue.<name>` — so you can filter, sort, and plot them directly in the MLflow store (the full table is always in `coefficients.json` too). By default this is the first `n_key_coefs=5` coefficients; pass `key_coefs="treat"` (or a list) to pin the ones you care about, since the treatment effect usually isn't simply the first coefficient (the intercept leads, and `C()`/`i()` expansions reorder).

In [ ]:
# the key-coefficient metrics logged by default, pulled back with the runs
coef_cols = sorted(c for c in runs.columns if c.startswith(("coef.", "se.", "pvalue.")))
runs[["vcov", *coef_cols]]

,vcov,coef.Intercept,coef.age,coef.income,coef.treat,pvalue.Intercept,pvalue.age,pvalue.income,pvalue.treat,se.Intercept,se.age,se.income,se.treat
0,hetero,0.482934,-0.018779,0.493218,1.978392,0.042369,0.000038,0.0,0.0,0.237299,0.004516,0.048326,0.092422
1,iid,0.482934,-0.018779,0.493218,1.978392,0.051242,0.000073,0.0,0.0,0.247129,0.004696,0.048483,0.093243


In [ ]:
# because they are real numeric metrics (not strings), the store can filter on them
mlflow.search_runs(
    experiment_names=["pyfixest-example"],
    filter_string="metrics.`coef.treat` > 0",
)[["params.vcov", "metrics.coef.treat", "metrics.pvalue.treat"]]

,params.vcov,metrics.coef.treat,metrics.pvalue.treat
0,hetero,1.978392,0.0
1,iid,1.978392,0.0


## Artifacts logged per run

Each run also logs the coefficient table (`coefficients.json`) and a human-readable regression table (`summary.md`), built from the run's own logged info.

In [ ]:
run_id = runs.loc[0, "run_id"]
[a.path for a in mlflow.artifacts.list_artifacts(run_id=run_id)]

['coefficients.json', 'summary.md']

## Render the logged regression table

The `summary.md` artifact is a self-built markdown regression table for that run. Load it back and render it inline.

In [ ]:
Markdown(mlflow.artifacts.load_text(f"runs:/{run_id}/summary.md"))

### outcome ~ treat + age + income

| Coefficient | Estimate | Std. Error | p-value |
|:---|---:|---:|---:|
| Intercept | 0.483* | 0.237 | 0.042 |
| treat | 1.978*** | 0.092 | 0.000 |
| age | -0.019*** | 0.005 | 0.000 |
| income | 0.493*** | 0.048 | 0.000 |

| Statistic | Value |
|:---|---:|
| nobs | 500 |
| r2 | 0.542 |
| adj_r2 | 0.539 |
| f_statistic | 192.927 |
| rmse | 1.029 |
| n_coefs | 4 |
| n_fes | 0 |

Significance: `*` p < 0.05, `**` p < 0.01, `***` p < 0.001. Cells: estimate with stars; standard error and p-value alongside.

## Coefficients across runs

`coeftable` reads each run's `coefficients.json` back into one long, self-describing frame (one row per run × coefficient, with the run's params joined on) — the quick way to get every coefficient across an experiment. The columns use plain snake_case names (`estimate`, `std_error`, `p_value`, `ci_low`/`ci_high` for the 95% CI, `t_value`).

In [ ]:
coefs = coeftable("pyfixest-example")
coefs[["coefficient", "estimate", "std_error", "p_value", "ci_low", "ci_high", "vcov"]]

,coefficient,estimate,std_error,p_value,ci_low,ci_high,vcov
0,Intercept,0.482934,0.237299,0.042369,0.016699,0.949168,hetero
1,treat,1.978392,0.092422,0.000000,1.796806,2.159978,hetero
2,age,-0.018779,0.004516,0.000038,-0.027651,-0.009906,hetero
3,income,0.493218,0.048326,0.000000,0.398269,0.588167,hetero
4,Intercept,0.482934,0.247129,0.051242,-0.002615,0.968483,iid
5,treat,1.978392,0.093243,0.000000,1.795193,2.161591,iid
6,age,-0.018779,0.004696,0.000073,-0.028005,-0.009553,iid
7,income,0.493218,0.048483,0.000000,0.397961,0.588475,iid


Filter to a single coefficient to compare it across runs (here, the treatment effect on `treat` under iid vs hetero errors):

In [ ]:
treat_rows = coeftable("pyfixest-example", coefficients="treat")
treat_rows[
    ["coefficient", "estimate", "std_error", "p_value", "ci_low", "ci_high", "vcov"]
]

,coefficient,estimate,std_error,p_value,ci_low,ci_high,vcov
0,treat,1.978392,0.092422,0.0,1.796806,2.159978,hetero
1,treat,1.978392,0.093243,0.0,1.795193,2.161591,iid


## Cross-run regression table

`etable` rebuilds a side-by-side comparison table — one column per run — entirely from the logged runs (coefficients, params, metrics). `type="md"` returns it as markdown.

In [ ]:
etable("pyfixest-example")

,(1),(2)
Intercept,0.483 (0.247),0.483* (0.237)
treat,1.978*** (0.093),1.978*** (0.092)
age,-0.019*** (0.005),-0.019*** (0.005)
income,0.493*** (0.048),0.493*** (0.048)
fml,outcome ~ treat + age + income,outcome ~ treat + age + income
vcov,iid,hetero
nobs,500,500
r2,0.542,0.542
adj_r2,0.539,0.539


### Show only some coefficients, or drop some

`coefficients` keeps only the rows you name; `drop` removes them. Handy for focusing a table on the treatment effect, or hiding the intercept and a block of controls.

In [ ]:
# keep only the treatment effect
etable("pyfixest-example", coefficients="treat")

,(1),(2)
treat,1.978*** (0.093),1.978*** (0.092)
fml,outcome ~ treat + age + income,outcome ~ treat + age + income
vcov,iid,hetero
nobs,500,500
r2,0.542,0.542
adj_r2,0.539,0.539


In [ ]:
# ... or show everything except the intercept
etable("pyfixest-example", drop="Intercept")

,(1),(2)
treat,1.978*** (0.093),1.978*** (0.092)
age,-0.019*** (0.005),-0.019*** (0.005)
income,0.493*** (0.048),0.493*** (0.048)
fml,outcome ~ treat + age + income,outcome ~ treat + age + income
vcov,iid,hetero
nobs,500,500
r2,0.542,0.542
adj_r2,0.539,0.539
